# Hospital Operations Analysis

---
## Sources & links 

Kaggle Dataset : [Hospital Management Dataset](https://www.kaggle.com/datasets/kanakbaghel/hospital-management-dataset?select=appointments.csv)

Website used to create ERD : 

---
## Overview 
- title 
- sources & links
- overview
- imports
    - library imports
    - data imports
- business understanding 
    - business objectives 
    - situation assesment 
    - project plan
- data understanding
    - data properties 
    - ERD
- methodoligy
- functions
- database manegement
    - clear old databse 
    - create databse connection
    - create tables
    - data insertion & validation
- business questions & exploritory data analysis
    - executive performance
    - branch performace 
    - operations
    - financial performance 
    - patient demographics 

---
## Imports

### Library Imports

In [1]:
import pandas as pd
import sqlite3
import matplotlib as plt
from pathlib import Path
from datetime import datetime

In [2]:
# this cell is to show the last update to this notebook
now = datetime.now()
print(f'This notebook was last updated at : {now} PST')

This notebook was last updated at : 2026-07-15 11:30:15.874415 PST


### Data Imports 

In [3]:
patients_df = pd.read_csv('operations_data/patients.csv')
doctors_df = pd.read_csv('operations_data/doctors.csv')
appointments_df = pd.read_csv('operations_data/appointments.csv')
billing_df = pd.read_csv('operations_data/billing.csv')
treatment_df = pd.read_csv('operations_data/treatments.csv')

In [4]:
patients_df.head()

,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com


In [5]:
doctors_df.head()

,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email
0,D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com
2,D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com
3,D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com
4,D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com


In [6]:
appointments_df.head()

,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled
1,A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show
2,A003,P048,D004,2023-06-28,8:00:00,Consultation,Cancelled
3,A004,P025,D006,2023-09-01,9:15:00,Consultation,Cancelled
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show


In [7]:
billing_df.head()

,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status
0,B001,P034,T001,2023-08-09,3941.97,Insurance,Pending
1,B002,P032,T002,2023-06-09,4158.44,Insurance,Paid
2,B003,P048,T003,2023-06-28,3731.55,Insurance,Paid
3,B004,P025,T004,2023-09-01,4799.86,Insurance,Failed
4,B005,P040,T005,2023-07-06,582.05,Credit Card,Pending


In [8]:
treatment_df.head()

,treatment_id,appointment_id,treatment_type,description,cost,treatment_date
0,T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09
1,T002,A002,MRI,Advanced protocol,4158.44,2023-06-09
2,T003,A003,MRI,Standard procedure,3731.55,2023-06-28
3,T004,A004,MRI,Basic screening,4799.86,2023-09-01
4,T005,A005,ECG,Standard procedure,582.05,2023-07-06


---
## Database Manegement 
Here we are firstly going to clear this notebooks databse, as this prevents data being du

In [9]:
# delete old database
db_path = Path('hospital.db')

if db_path.exists():
    db_path.unlink()

Then create an empty databse for us to add tables and eventually populate those tables with the data from the `.csv` file. This is done this way so we can have much more control over the database. 

In [10]:
conn = sqlite3.connect('hospital.db')
conn.execute('PRAGMA foreign_keys = ON')

### Tables
Here we create empty tables with the constraints we want for each data point. We will also be creating smaller dataframes for each table to pull data from so we can only use the data from each table we want. 

In [11]:
# patients table
conn.execute("""
CREATE TABLE patients (
patient_id TEXT PRIMARY KEY,
first_name TEXT,
last_name TEXT,
gender TEXT,
date_of_birth TEXT
);
""")

In [12]:
patients_sql = patients_df[['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth']]

In [13]:
patients_sql.head()

,patient_id,first_name,last_name,gender,date_of_birth
0,P001,David,Williams,F,1955-06-04
1,P002,Emily,Smith,F,1984-10-12
2,P003,Laura,Jones,M,1977-08-21
3,P004,Michael,Johnson,F,1981-02-20
4,P005,David,Wilson,M,1960-06-23


In [14]:
# doctors table
conn.execute("""
CREATE TABLE doctors (
doctor_id TEXT PRIMARY KEY,
first_name TEXT,
last_name TEXT,
specialization TEXT,
hospital_branch TEXT
);
""")

In [15]:
doctors_sql = doctors_df [['doctor_id', 'first_name', 'last_name', 'specialization', 'hospital_branch']]

In [16]:
doctors_sql.head()

,doctor_id,first_name,last_name,specialization,hospital_branch
0,D001,David,Taylor,Dermatology,Westside Clinic
1,D002,Jane,Davis,Pediatrics,Eastside Clinic
2,D003,Jane,Smith,Pediatrics,Eastside Clinic
3,D004,David,Jones,Pediatrics,Central Hospital
4,D005,Sarah,Taylor,Dermatology,Central Hospital


In [17]:
# appointmets table
conn.execute("""
CREATE TABLE appointments (
appointment_id TEXT PRIMARY KEY,
patient_id TEXT,
doctor_id TEXT,
appointment_date TEXT,
status TEXT,
reason_for_visit TEXT,

FOREIGN KEY(patient_id)
    REFERENCES patients(patient_id),
FOREIGN KEY(doctor_id)
    REFERENCES doctors(doctor_id)
);
""")

In [18]:
appointments_sql = appointments_df [['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'status', 'reason_for_visit']]

In [19]:
appointments_sql.head()

,appointment_id,patient_id,doctor_id,appointment_date,status,reason_for_visit
0,A001,P034,D009,2023-08-09,Scheduled,Therapy
1,A002,P032,D004,2023-06-09,No-show,Therapy
2,A003,P048,D004,2023-06-28,Cancelled,Consultation
3,A004,P025,D006,2023-09-01,Cancelled,Consultation
4,A005,P040,D003,2023-07-06,No-show,Emergency


In [20]:
# billing table
conn.execute("""
CREATE TABLE billing (
bill_id TEXT PRIMARY KEY,
patient_id TEXT,
treatment_id TEXT,
bill_date TEXT,
amount REAL,

FOREIGN KEY(patient_id)
    REFERENCES patients(patient_id),
FOREIGN KEY(treatment_id)
    REFERENCES treatments(treatment_id)
);
""")

In [21]:
billing_sql = billing_df[['bill_id', 'patient_id', 'treatment_id', 'bill_date', 'amount']]

In [22]:
billing_sql.head()

,bill_id,patient_id,treatment_id,bill_date,amount
0,B001,P034,T001,2023-08-09,3941.97
1,B002,P032,T002,2023-06-09,4158.44
2,B003,P048,T003,2023-06-28,3731.55
3,B004,P025,T004,2023-09-01,4799.86
4,B005,P040,T005,2023-07-06,582.05


In [23]:
# treatment table
conn.execute("""
CREATE TABLE treatments (
treatment_id TEXT PRIMARY KEY,
appointment_id TEXT NOT NULL,
treatment_type TEXT,
description TEXT,
treatment_date TEXT,

FOREIGN KEY(appointment_id)
    REFERENCES appointments(appointment_id)
);
""")

In [24]:
treatment_sql = treatment_df[['treatment_id', 'appointment_id', 'treatment_type', 'description', 'treatment_date']]

In [25]:
treatment_sql.head()

,treatment_id,appointment_id,treatment_type,description,treatment_date
0,T001,A001,Chemotherapy,Basic screening,2023-08-09
1,T002,A002,MRI,Advanced protocol,2023-06-09
2,T003,A003,MRI,Standard procedure,2023-06-28
3,T004,A004,MRI,Basic screening,2023-09-01
4,T005,A005,ECG,Standard procedure,2023-07-06


### Data Insertion
Here we are populating our tables, then using the output below the insertion to validate it worked properly.

In [26]:
# patients table
patients_sql.to_sql(
    'patients',
    conn,
    if_exists = 'append',
    index = False
)

50

In [27]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_patients
FROM patients
""", conn)

,total_patients
0,50


In [28]:
pd.read_sql("""
SELECT *
FROM patients
LIMIT 5
""", conn)

,patient_id,first_name,last_name,gender,date_of_birth
0,P001,David,Williams,F,1955-06-04
1,P002,Emily,Smith,F,1984-10-12
2,P003,Laura,Jones,M,1977-08-21
3,P004,Michael,Johnson,F,1981-02-20
4,P005,David,Wilson,M,1960-06-23


In [29]:
# doctors table
doctors_sql.to_sql(
    'doctors',
    conn,
    if_exists = 'append',
    index = False
)

10

In [30]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_doctors
FROM doctors
""", conn)

,total_doctors
0,10


In [31]:
pd.read_sql("""
SELECT * 
FROM doctors
LIMIT 5
""", conn)

,doctor_id,first_name,last_name,specialization,hospital_branch
0,D001,David,Taylor,Dermatology,Westside Clinic
1,D002,Jane,Davis,Pediatrics,Eastside Clinic
2,D003,Jane,Smith,Pediatrics,Eastside Clinic
3,D004,David,Jones,Pediatrics,Central Hospital
4,D005,Sarah,Taylor,Dermatology,Central Hospital


In [32]:
# appointments table
appointments_sql.to_sql(
    'appointments',
    conn,
    if_exists = 'append',
    index = False
)

200

In [33]:
# validation
pd.read_sql("""
SELECT COUNT(*) as total_appointments
FROM appointments
""", conn)

,total_appointments
0,200


In [34]:
pd.read_sql("""
select *
FROM appointments
LIMIT 5
""", conn)

,appointment_id,patient_id,doctor_id,appointment_date,status,reason_for_visit
0,A001,P034,D009,2023-08-09,Scheduled,Therapy
1,A002,P032,D004,2023-06-09,No-show,Therapy
2,A003,P048,D004,2023-06-28,Cancelled,Consultation
3,A004,P025,D006,2023-09-01,Cancelled,Consultation
4,A005,P040,D003,2023-07-06,No-show,Emergency


In [35]:
# treatments table
treatment_sql.to_sql(
    'treatments',
    conn,
    if_exists = 'append', 
    index = False
)

200

In [36]:
# validation
pd.read_sql("""
select COUNT(*) as total_treatments
from treatments
""", conn)

,total_treatments
0,200


In [37]:
pd.read_sql("""
select *
from treatments
limit 5
""", conn)

,treatment_id,appointment_id,treatment_type,description,treatment_date
0,T001,A001,Chemotherapy,Basic screening,2023-08-09
1,T002,A002,MRI,Advanced protocol,2023-06-09
2,T003,A003,MRI,Standard procedure,2023-06-28
3,T004,A004,MRI,Basic screening,2023-09-01
4,T005,A005,ECG,Standard procedure,2023-07-06


In [38]:
# billing table 
billing_sql.to_sql(
    'billing',
    conn,
    if_exists = 'append',
    index = False
)

200

In [39]:
# validation
pd.read_sql("""
select COUNT(*) as total_bills
from billing
""", conn)

,total_bills
0,200


In [40]:
pd.read_sql("""
select *
from billing
limit 5
""", conn)

,bill_id,patient_id,treatment_id,bill_date,amount
0,B001,P034,T001,2023-08-09,3941.97
1,B002,P032,T002,2023-06-09,4158.44
2,B003,P048,T003,2023-06-28,3731.55
3,B004,P025,T004,2023-09-01,4799.86
4,B005,P040,T005,2023-07-06,582.05


In [41]:
conn.commit()

---
## Business Questions & Exploritory Data Analysis

### Executive Performance 

In [42]:
# what is total revenue

pd.read_sql("""
select sum(amount) as total_income
from billing
""", conn)

,total_income
0,551249.85


In [43]:
# how many patients were treated
pd.read_sql("""
select count(*) as total_patients
from patients
""", conn)

,total_patients
0,50
